# 04 - Champion Selection & Experiment Balance Check

This notebook:
1. Dynamically queries `data/db.sqlite3` for all completed LLaMEA experiments present in the database.
2. Displays an **experiment summary** grouped by problem ID, dimension, noise level, and prompt strategy.
3. Selects separate **Clean** (`noise_std = 0.0`) and **Noisy** (`noise_std > 0.0`) champion algorithms per problem (lowest `final_error` across iterations).
4. Exports `data/champions.json` for evaluation in Notebook 05.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT
from infra.storage import get_db_connection, get_db_engine

CHAMPIONS_PATH = DATA_DIR / 'champions.json'
print(f'Champions Output Path: {CHAMPIONS_PATH}')


Database Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/db.sqlite3
Champions Output Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json


## 1. Experiment Balance Check

In [2]:
# Query all completed experiments dynamically
query_exps = """
SELECT 
    id as exp_id,
    problem_id,
    dim,
    noise_std,
    prompt_strategy,
    llm_name,
    status,
    best_algorithm,
    best_final_error
FROM experiments
WHERE status = 'completed'
ORDER BY problem_id, dim, noise_std, exp_id
"""
with get_db_connection() as conn:
    df_exps = pd.read_sql_query(query_exps, conn)

print(f'Total completed experiments in database: {len(df_exps)}')

if not df_exps.empty:
    print('\n=== Completed Experiments Summary ===')
    summary = df_exps.groupby(['problem_id', 'dim', 'noise_std', 'prompt_strategy']).size().reset_index(name='count')
    print(summary.to_string(index=False))
else:
    print('WARNING: No completed experiments found in database.')


Total completed experiments in database: 42

=== Completed Experiments Summary ===
 problem_id  dim  noise_std prompt_strategy  count
          1    2       0.00        baseline      1
          1    2       0.05        baseline      1
          1    3       0.00        baseline      3
          1    3       0.05        baseline      3
          8    2       0.00        baseline      1
          8    2       0.05        baseline      1
          8    3       0.00        baseline      3
          8    3       0.05        baseline      3
         11    2       0.00        baseline      1
         11    2       0.05        baseline      1
         11    3       0.00        baseline      3
         11    3       0.05        baseline      3
         15    2       0.00        baseline      2
         15    2       0.05        baseline      2
         15    3       0.00        baseline      3
         15    3       0.05        baseline      3
         21    2       0.00        baseline      1

## 2. Select Problem-Specific Champions (Dynamically)

In [3]:
# Query all iterations from completed experiments to find true absolute lowest final_error per problem & mode
iter_query = """
SELECT 
    e.problem_id,
    e.dim,
    e.noise_std,
    e.id as experiment_id,
    e.llm_name,
    e.prompt_strategy,
    i.id as iteration_id,
    i.algorithm_name,
    i.final_error,
    i.evaluations_used,
    i.code_path
FROM iterations i
JOIN experiments e ON i.experiment_id = e.id
WHERE e.status = 'completed'
  AND i.final_error IS NOT NULL
ORDER BY i.final_error ASC
"""
with get_db_connection() as conn:
    df_iters = pd.read_sql_query(iter_query, conn)

champions = {}

print('=== Clean & Noisy Champions (Dynamically Selected) ===')
if df_iters.empty:
    print('No completed iterations found in the database.')
else:
    for p_id in sorted(df_iters['problem_id'].unique()):
        p_subset = df_iters[df_iters['problem_id'] == p_id]
        
        # 1. Clean Champion (noise_std == 0.0)
        clean_subset = p_subset[p_subset['noise_std'] == 0.0]
        if not clean_subset.empty:
            best_clean = clean_subset.iloc[0]
            key_clean = f'f{p_id}_clean'
            champions[key_clean] = {
                'problem_id': int(p_id),
                'mode': 'clean',
                'noise_std': 0.0,
                'dim': int(best_clean['dim']),
                'experiment_id': int(best_clean['experiment_id']),
                'algorithm_name': str(best_clean['algorithm_name']),
                'final_error': float(best_clean['final_error']),
                'evaluations_used': int(best_clean['evaluations_used']) if pd.notnull(best_clean['evaluations_used']) else None,
                'code_path': str(best_clean['code_path']),
                'llm_name': str(best_clean['llm_name']),
                'prompt_strategy': str(best_clean['prompt_strategy'])
            }
            print(f"f{p_id} Clean: {best_clean['algorithm_name']} (Exp #{best_clean['experiment_id']}) -> final_error = {best_clean['final_error']:.6e}")
        
        # 2. Noisy Champion (noise_std > 0.0)
        noisy_subset = p_subset[p_subset['noise_std'] > 0.0]
        if not noisy_subset.empty:
            best_noisy = noisy_subset.iloc[0]
            key_noisy = f'f{p_id}_noisy'
            champions[key_noisy] = {
                'problem_id': int(p_id),
                'mode': 'noisy',
                'noise_std': float(best_noisy['noise_std']),
                'dim': int(best_noisy['dim']),
                'experiment_id': int(best_noisy['experiment_id']),
                'algorithm_name': str(best_noisy['algorithm_name']),
                'final_error': float(best_noisy['final_error']),
                'evaluations_used': int(best_noisy['evaluations_used']) if pd.notnull(best_noisy['evaluations_used']) else None,
                'code_path': str(best_noisy['code_path']),
                'llm_name': str(best_noisy['llm_name']),
                'prompt_strategy': str(best_noisy['prompt_strategy'])
            }
            print(f"f{p_id} Noisy (std={best_noisy['noise_std']}): {best_noisy['algorithm_name']} (Exp #{best_noisy['experiment_id']}) -> final_error = {best_noisy['final_error']:.6e}")

# Save champions.json
CHAMPIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHAMPIONS_PATH, 'w') as f:
    json.dump(champions, f, indent=2)

print(f'\nExported {len(champions)} champion(s) to {CHAMPIONS_PATH}')


=== Clean & Noisy Champions (Dynamically Selected) ===
f1 Clean: ImprovedHillClimbing (Exp #13) -> final_error = 0.000000e+00
f1 Noisy (std=0.05): ImprovedRandomSearchOptimizer (Exp #2) -> final_error = 0.000000e+00
f8 Clean: GradientDescentOptimizer (Exp #4) -> final_error = 8.471250e+03
f8 Noisy (std=0.05): ImprovedEvolutionaryAlgorithm (Exp #16) -> final_error = 5.596235e-11
f11 Clean: ImprovedOptimizer (Exp #38) -> final_error = 0.000000e+00
f11 Noisy (std=0.05): AdvancedEvolutionaryStrategy (Exp #6) -> final_error = 5.200865e-05
f15 Clean: AdaptiveMutationOptimizer (Exp #8) -> final_error = 2.494783e-03
f15 Noisy (std=0.05): MultiObjectiveGA (Exp #30) -> final_error = 1.342386e-03
f21 Clean: AdaptiveGradientSearch (Exp #9) -> final_error = 4.263256e-14
f21 Noisy (std=0.05): CMAES (Exp #10) -> final_error = 5.848761e-10

Exported 10 champion(s) to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json
